# Laya fine-tune on Google Colab — Persian System One decision model

Fine-tunes the **open-weights [Laya](https://github.com/NandhaKishorM/laya)**
System One decision model (the `multilingual` mmBERT-base checkpoint) on this
project's Persian detection data, adapting the reference recipe from the laya
repo (`notebooks/laya_finetune_typed_decisions_mps.py`, originally written for
Apple Silicon / Kaggle 2xT4).

It drives the repo's own `training/train_laya.py` — the same script the local
CPU run uses — with GPU-appropriate flags (last 4 encoder layers unfrozen,
bigger micro-batch, the full 6000-record train split, 3 epochs, RLCD loop +
per-question-type temperature calibration + held-out eval).

**What you need on Google Drive first** (see cell 2): the `aitextd/` package
folder, three `training/*.py` files, and the two `system_one_*.jsonl` files.

**Output:** `models/laya_persian` (a standard Laya checkpoint directory) plus
`metrics.json`, copied back to Drive under `AITextDetection/trained/`.

**Runtime:** ~5–10 min setup + download (680 MB checkpoint), then roughly
45–90 min per epoch on a T4 — pick **Runtime → Change runtime type → T4 GPU**
before running.

In [ ]:
!pip install -q laya

import torch, transformers, laya
print("torch:", torch.__version__, "| transformers:", transformers.__version__, "| laya:", laya.__version__ if hasattr(laya, "__version__") else "n/a")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch runtime to T4 GPU")

## 1. Mount Drive and stage the repo files

The notebook rebuilds just enough of the repo layout under `/content/AITextDetection`
(the script resolves every path through `aitextd.paths`, so nothing is hardcoded).
Put these in **`MyDrive/AITextDetection/`** before running:

```
MyDrive/AITextDetection/
├── aitextd/                        ← the whole aitextd package folder
└── training/
    ├── train_laya.py               ← the fine-tuning script this notebook drives
    ├── eval_online_systemone.py    ← exports the production QUESTIONS dict
    ├── split.py                    ← shared RANDOM_SEED / split helpers
    └── system_one_data/
        ├── system_one_train.jsonl  ← 53 MB, GPT-2-distilled soft targets
        └── system_one_test.jsonl   ← 13 MB held-out split
```
**Fastest option:** upload the single `colab_laya_files.zip` (repo root) to
`MyDrive/` and run `!unzip -o /content/drive/MyDrive/colab_laya_files.zip -d
/content/drive/MyDrive/` — it extracts exactly this layout. Or upload the
files individually as listed below.


In [ ]:
import os, shutil, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/AITextDetection"
REPO = "/content/AITextDetection"

NEEDED = [
    f"{DRIVE_DIR}/aitextd/__init__.py",
    f"{DRIVE_DIR}/training/train_laya.py",
    f"{DRIVE_DIR}/training/eval_online_systemone.py",
    f"{DRIVE_DIR}/training/split.py",
    f"{DRIVE_DIR}/training/system_one_data/system_one_train.jsonl",
    f"{DRIVE_DIR}/training/system_one_data/system_one_test.jsonl",
]
missing = [p for p in NEEDED if not os.path.exists(p)]
if missing:
    raise SystemExit("Upload these to Drive first:\n  " + "\n  ".join(missing))

os.makedirs(f"{REPO}/training/system_one_data", exist_ok=True)
shutil.copytree(f"{DRIVE_DIR}/aitextd", f"{REPO}/aitextd", dirs_exist_ok=True)
for name in ("train_laya.py", "eval_online_systemone.py", "split.py"):
    shutil.copy2(f"{DRIVE_DIR}/training/{name}", f"{REPO}/training/{name}")
for name in ("system_one_train.jsonl", "system_one_test.jsonl"):
    shutil.copy2(f"{DRIVE_DIR}/training/system_one_data/{name}",
                 f"{REPO}/training/system_one_data/{name}")
print("Staged repo at", REPO)
!ls {REPO} {REPO}/training {REPO}/training/system_one_data

## 2. Fine-tune

Flags (GPU-appropriate, vs. the CPU defaults in `train_laya.py`):

| flag | value | why |
|---|---|---|
| `--device cuda` | T4 | auto-detect would also pick it, stated explicitly |
| `--unfrozen-layers 4` | last 4 of 22 mmBERT layers | the CPU run froze the whole encoder; on GPU a real partial fine-tune is affordable |
| `--micro-batch 16 --grad-accum 2` | effective batch 32 | same effective batch as the reference recipe |
| `--records 6000 --calib-records 500 --test-records 1500` | full splits | all train records (500 held out for temperature fitting), the whole test split |

The multilingual checkpoint (`convaiinnovations/laya`, subfolder `multilingual`)
downloads automatically (~680 MB) on the first run. Item preprocessing is cached,
so an interrupted run re-starts cheaply. To continue from an earlier run,
add `--init-from <checkpoint dir>` (e.g. a Drive copy of `trained/laya_persian`).

Quick smoke first (optional): `--records 40 --calib-records 8 --test-records 8
--epochs 1` finishes in a couple of minutes and exercises the whole path.

In [ ]:
import os, subprocess, sys

REPO = "/content/AITextDetection"
cmd = [
    sys.executable, "training/train_laya.py",
    "--device", "cuda",
    "--unfrozen-layers", "4",
    "--micro-batch", "16",
    "--grad-accum", "2",
    "--epochs", "3",
    # --records 40, --calib-records 8, --test-records 8, --epochs 1,   # smoke
    "--records", "6000",
    "--calib-records", "500",
    "--test-records", "1500",
]
result = subprocess.run(cmd, cwd=REPO,
                        env={**os.environ, "TOKENIZERS_PARALLELISM": "false"})
if result.returncode != 0:
    raise SystemExit(f"training failed with exit code {result.returncode} — see the log above")

## 3. Save the checkpoint back to Drive

`models/laya_persian` holds a standard Laya checkpoint
(`model.safetensors`, `rl_agent_config.json` with the fitted temperatures,
`encoder/`, `tokenizer/`) plus `metrics.json` and `checkpoint_meta.json`.

In [ ]:
import json, shutil

REPO = "/content/AITextDetection"
OUT = f"{REPO}/models/laya_persian"
DEST = "/content/drive/MyDrive/AITextDetection/trained/laya_persian"

metrics = json.load(open(f"{OUT}/metrics.json"))
print("metrics:", json.dumps(metrics, indent=2))

shutil.copytree(OUT, DEST, dirs_exist_ok=True)
print("Checkpoint copied to", DEST)

## 4. Verify: answer the production questions

Loads the fine-tuned checkpoint exactly the way a serving process would
(`laya.load`) and asks the same three typed questions the web app sends to
System One providers (`source` choice / `machine_likeness` score /
`needs_review` noul).

In [ ]:
import json, sys

REPO = "/content/AITextDetection"
sys.path.insert(0, f"{REPO}/training")

from laya import load
from eval_online_systemone import QUESTIONS

agent = load(f"{REPO}/models/laya_persian", device="cuda")

samples = {
    "AI-like": ("با توجه به بررسی‌های انجام‌شده، لازم است به نکات زیر اشاره شود. "
                "این روش دارای مزایای متعددی است و می‌تواند در آینده نیز مورد استفاده قرار گیرد."),
    "human-like": ("وای چه روزی بود! از صبح داشتم می‌دویدم که دیر نکنم، بعد هم تا رسیدم "
                   "صندلی خالی نبود. خلاصه که حسابی خسته شدم ولی ارزشش رو داشت."),
}
for label, text in samples.items():
    r = agent.predict({"text": text}, QUESTIONS)
    src = r["answers"]["source"]
    print(f"{label}: ai_generated={src['probabilities']['ai_generated']:.3f} "
          f"(choice={src['choice']}) "
          f"score={r['answers']['machine_likeness']['score']:.2f} "
          f"noul={r['answers']['needs_review']['noul']:.3f}")